<h3>HWP와 HWPX의 경계</h3>
<p><strong>개념</strong> HWPX는 ZIP 컨테이너 안에 XML을 담습니다. 이 실습은 패키지 내부의 본문 표식을 검사하며 한컴오피스 호환성이나 레거시 HWP 파싱을 보장하지 않습니다.</p>

In [1]:
import sys
print(sys.version.split()[0])


3.12.13


In [2]:
from pathlib import Path
from xml.etree import ElementTree as ET
import zipfile

roots = [Path.cwd(), *Path.cwd().parents]
labs = [p / "labs/part3/ch10-document-processing" for p in roots
        if (p / "labs/part3/ch10-document-processing").is_dir()]
if not labs: raise FileNotFoundError("저장소 루트에서 노트북을 실행하세요")
lab = labs[0]
hwpx = lab / "fixtures/minimal-marker.hwpx"


<h3>ZIP과 XML 직접 검사</h3>
<p><strong>확인할 점</strong> <code>Contents/section*.xml</code>만 읽고 태그의 로컬 이름이 <code>t</code>인 텍스트를 추출합니다.</p>

In [3]:
with zipfile.ZipFile(hwpx) as archive:
    sections = sorted(n for n in archive.namelist()
                      if n.startswith("Contents/section") and n.endswith(".xml"))
    texts = []
    for name in sections:
        root = ET.fromstring(archive.read(name))
        texts.append("".join(n.text or "" for n in root.iter()
                             if n.tag.rsplit("}", 1)[-1] == "t"))
print({"sections": sections, "text": texts})


{'sections': ['Contents/section0.xml'], 'text': ['한글 문서 처리 실습입니다.']}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 레거시 HWP는 ZIP이 아닙니다. 확장자만 보고 HWPX와 같은 방식으로 열면 오류가 납니다.</p>

In [4]:
legacy = lab / "fixtures/legacy-signature.hwp"
try:
    with zipfile.ZipFile(legacy) as archive:
        print(archive.namelist())
except zipfile.BadZipFile as error:
    print(f"caught: legacy HWP needs a dedicated parser ({error})")


caught: legacy HWP needs a dedicated parser (File is not a zip file)


<p><strong>수정 이유</strong> 파일 시그니처를 먼저 검사합니다. HWPX는 ZIP/XML 경로로 보내고 레거시 HWP는 지원하지 않는 형식으로 명시합니다.</p>

In [5]:
ole_signature = bytes.fromhex("D0CF11E0A1B11AE1")
results = {}
for path in (legacy, hwpx):
    if path.read_bytes()[:8] == ole_signature:
        results[path.suffix] = "전용 파서가 필요한 미지원 HWP"
        continue
    if not zipfile.is_zipfile(path):
        raise ValueError("HWPX ZIP 형식이 아닙니다.")
    with zipfile.ZipFile(path) as archive:
        name = next(n for n in archive.namelist() if n.startswith("Contents/section"))
        root = ET.fromstring(archive.read(name))
        results[path.suffix] = "".join(n.text or "" for n in root.iter() if n.tag.rsplit("}", 1)[-1] == "t")
assert results[".hwp"] == "전용 파서가 필요한 미지원 HWP" and results[".hwpx"] == texts[0]
print(results)


{'.hwp': '전용 파서가 필요한 미지원 HWP', '.hwpx': '한글 문서 처리 실습입니다.'}
